# AeroNet Lite Notebook 2 — Drone Anomaly Classification

This notebook is the ML classification part of the project. It classifies drone flight conditions into:

- `Normal`
- `Battery anomaly`
- `Route anomaly`
- `Sensor spike`

Synthetic labels are used because the project brief allows synthetic anomaly rules for a 2-week implementation.

In [ ]:
# Run this once if widgets do not appear:
# pip install ipywidgets

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from ipywidgets import interact, FloatSlider, Dropdown

from ml_pipeline import make_synthetic_anomaly_data, train_anomaly_model, predict_anomaly

## 1. Load synthetic anomaly telemetry

Each row represents one flight-status sample. The label is generated using transparent rules.

In [ ]:
df = make_synthetic_anomaly_data(n=900, seed=12)
display(df.head(10))
display(df["label"].value_counts().to_frame("count"))
display(df.describe().round(2))

## 2. Interactive feature scatter plot

Change the X/Y features to see how anomaly classes separate.

In [ ]:
def plot_pair(x_feature, y_feature):
    plt.figure(figsize=(7, 5))
    for label, part in df.groupby("label"):
        plt.scatter(part[x_feature], part[y_feature], alpha=0.55, label=label)
    plt.xlabel(x_feature)
    plt.ylabel(y_feature)
    plt.title(f"{y_feature} vs {x_feature}")
    plt.legend()
    plt.grid(True, alpha=0.25)
    plt.show()

feature_options = ["battery_drop", "speed", "route_deviation", "altitude_change"]
interact(
    plot_pair,
    x_feature=Dropdown(options=feature_options, value="battery_drop", description="X"),
    y_feature=Dropdown(options=feature_options, value="route_deviation", description="Y"),
);

## 3. Train classifier and show confusion matrix

Try Decision Tree and Random Forest. Decision Tree is easier to explain; Random Forest is usually more stable.

In [ ]:
def train_and_show(model_name):
    result = train_anomaly_model(model_name=model_name, seed=12)
    display(Markdown(f"### {result.model_name}"))
    print("Accuracy:", result.accuracy)
    display(result.sample_frame.head(12))

    cm = result.confusion
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm)
    ax.set_xticks(range(len(result.labels)))
    ax.set_yticks(range(len(result.labels)))
    ax.set_xticklabels(result.labels, rotation=35, ha="right")
    ax.set_yticklabels(result.labels)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    ax.set_title("Confusion matrix")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

    if hasattr(result.model, "feature_importances_"):
        importance = pd.Series(result.model.feature_importances_, index=result.feature_names).sort_values(ascending=False)
        plt.figure(figsize=(7, 4))
        importance.plot(kind="bar")
        plt.title("Feature importance")
        plt.ylabel("Importance")
        plt.grid(True, axis="y", alpha=0.25)
        plt.show()

interact(
    train_and_show,
    model_name=Dropdown(options=["Decision Tree", "Random Forest"], value="Random Forest", description="Model"),
);

## 4. Interactive anomaly prediction

Move the sliders and see which anomaly class the trained classifier predicts.

In [ ]:
final_anomaly_result = train_anomaly_model(model_name="Random Forest", seed=12)

def classify_sample(battery_drop, speed, route_deviation, altitude_change):
    label = predict_anomaly(
        final_anomaly_result.model,
        battery_drop=battery_drop,
        speed=speed,
        route_deviation=route_deviation,
        altitude_change=altitude_change,
    )
    print("Predicted class:", label)
    if label != "Normal":
        print("Suggested simulator action: raise alert, reroute, or force drone to return to hub.")
    else:
        print("Suggested simulator action: continue route normally.")

interact(
    classify_sample,
    battery_drop=FloatSlider(min=0, max=20, step=0.2, value=3.0, description="Battery drop"),
    speed=FloatSlider(min=0, max=25, step=0.2, value=8.0, description="Speed"),
    route_deviation=FloatSlider(min=0, max=8, step=0.1, value=0.5, description="Deviation"),
    altitude_change=FloatSlider(min=0, max=15, step=0.2, value=1.0, description="Altitude"),
);

## Viva explanation

- This is a **classification** problem because the output is a category.
- The confusion matrix shows which classes are being confused.
- `battery_drop` is strongly linked to battery anomalies.
- `route_deviation` is strongly linked to route anomalies.
- `speed` and `altitude_change` are useful for sensor-spike behavior.